In [16]:
import os
import mlflow
import json
from mlflow import MlflowClient
from minio import Minio


In [3]:
MLFLOW_TRACKING_URI = "http://mlflow-server.mlflow.svc.cluster.local:8080"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print("Mlflow URI:", mlflow.get_tracking_uri())

Mlflow URI: http://mlflow-server.mlflow.svc.cluster.local:8080


In [5]:
# Find the latest run
experiment = client.get_experiment_by_name("openshift-ai-drift-blog-1")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=1
)
run = runs[0]
print("Run ID:", run.info.run_id)
print("Artifact URI:", run.info.artifact_uri)


Run ID: 4a9f6eb346924ef2aa83f93d71e1c617
Artifact URI: mlflow-artifacts:/4/4a9f6eb346924ef2aa83f93d71e1c617/artifacts


In [13]:
# Let's download serving artifact from MLflow
local_serving_model = mlflow.artifacts.download_artifacts(
    run_id=run.info.run_id,
    artifact_path="deployment/sklearn/iris_model.pkl",
    dst_path="serving-download"
)
print("Downloaded to:", local_serving_model)
import shutil
shutil.copy("models/iris_model.pkl","models/model.joblib")
    

Downloaded to: /opt/app-root/src/serving-download/iris_model.pkl


'models/model.joblib'

In [7]:
# Configure MinIO for OpenShift AI serving
MINIO_ENDPOINT = "minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"

client_minio = Minio(
    MINIO_ENDPOINT,
    access_key= MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)



In [9]:
bucket = "mlops-models"
if not client_minio.bucket_exists(bucket):
    client_minio.make_bucket(bucket)

print("Bucket ready:", bucket)

Bucket ready: mlops-models


In [17]:
settings = {
    "name": "iris",
    "implementation": "mlserver_sklearn.SKLearnModel",
    "parameters": {
        "uri": "./iris_model.pkl"
    }
}
with open("models/model-settings.json", "w") as f:
    json.dump(settings, f, indent=2)

In [20]:
# Upload model and model.json to serving path
object_name = "iris-model/kserve/iris_model.pkl"
client_minio.fput_object(
    bucket,
    object_name,
    local_serving_model
)
object_json_name = "iris-model/kserve/model-settings.json"
client_minio.fput_object(
    bucket,
    object_json_name,
    "models/model-settings.json"
)


ObjectWriteResult(bucket_name='mlops-models', object_name='iris-model/kserve/model-settings.json', version_id=None, etag='4a786c42e2304f44b969ac808f8eba10', http_headers=HTTPHeaderDict({'Accept-Ranges': 'bytes', 'Content-Length': '0', 'ETag': '"4a786c42e2304f44b969ac808f8eba10"', 'Server': 'MinIO', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Vary': 'Origin, Accept-Encoding', 'X-Amz-Id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'X-Amz-Request-Id': '18BE32663109E0F1', 'X-Content-Type-Options': 'nosniff', 'X-Ratelimit-Limit': '304', 'X-Ratelimit-Remaining': '304', 'X-Xss-Protection': '1; mode=block', 'Date': 'Wed, 01 Jul 2026 14:58:37 GMT'}), last_modified=None, location=None)

In [21]:
#Verify upload
objects = client_minio.list_objects(
    bucket,
    prefix="iris-model/kserve/",
    recursive=True
)
for obj in objects:
    print(obj.object_name, obj.size)

iris-model/kserve/iris_model.pkl 168065
iris-model/kserve/model-settings.json 126
